In [ ]:
%load_ext autoreload
%autoreload 2
%cd /opt/tiger/samantha

In [ ]:
from pyarrow.parquet import ParquetFile
from lightning_fabric.utilities.cloud_io import get_filesystem

def read_parquet(fp: str) -> ParquetFile:
    fs = get_filesystem(fp)
    stream = fs.open(fp, skip_instance_cache=True)
    return ParquetFile(stream)

In [ ]:
from tqdm import tqdm

# parquet_file_audio = read_parquet("hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data_store/BigMusic/music_Ssstk-pond5_Mnonvocal_T44k_N1608k/data/part=00004/shard_00120.parquet")
# parquet_file_feature = read_parquet("hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data_store/BigMusic/music_Ssstk-pond5_Mnonvocal_T44k_N1608k_AudioCodec_f81b3fa_64l/data/part=00004/shard_00120.parquet")


# parquet_file_audio = read_parquet("hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data_store/BigMusic/music_Ssstk-pond5_Mnonvocal_T44k_N1608k/data/part=00004/shard_00120.parquet")
# parquet_file_feature = read_parquet("hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data_store/BigMusic/music_Ssstk-pond5_Mnonvocal_T44k_N1608k_AudioCodec_7c355ea_64l/data/part=00004/shard_00120.parquet")


parquet_file_audio = read_parquet("hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data_store/BigMusic/music_everynoise_N937k_Lmix_mp3/data/shard_02505.parquet")
parquet_file_feature = read_parquet("hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data_store/BigMusic/music_everynoise_N937k_Lmix_mp3_AudioCodec_7c355ea_64l/data/shard_02505.parquet")


for row_group in tqdm(range(parquet_file_audio.num_row_groups)):
    audio_data = parquet_file_audio.read_row_group(row_group).to_pandas()
    feature_data = parquet_file_feature.read_row_group(row_group).to_pandas()
    
    assert (audio_data["uttid"].tolist() == feature_data["uttid"].tolist()), (audio_data["uttid"].tolist(), feature_data["uttid"].tolist())
    # print(audio_data["uttid"].tolist(), feature_data["uttid"].tolist())

In [ ]:
import torch
import random
from io import BytesIO
from samantha.data.av_audio import audio_read
from IPython.display import display, Audio


def sample_from_dfs():
    assert parquet_file_audio.num_row_groups == parquet_file_feature.num_row_groups
    
    row_group_idx = random.randint(0, parquet_file_feature.num_row_groups)
        
    audio_data = parquet_file_audio.read_row_group(row_group_idx).to_pandas()
    feature_data = parquet_file_feature.read_row_group(row_group_idx).to_pandas()
    
    assert len(audio_data) == len(feature_data)
    
    group_idx = random.randint(0, len(audio_data) - 1)    
     
    audio_bytes = audio_data.loc[group_idx, "audio"]
    feature_bytes = feature_data.loc[group_idx, "feature"]
    
    audio = audio_read(BytesIO(audio_bytes), format="wav")
    feature = torch.load(BytesIO(feature_bytes))
    return audio, feature
    

In [ ]:
from recipes.research.audio_codec.zoo import AudioCodec_f81b3fa_64l, AudioCodec_7c355ea_64l
audio_codec = AudioCodec_7c355ea_64l()
audio_codec = audio_codec.to("cuda")
audio_codec = audio_codec.eval()

In [ ]:
from io import BytesIO
from samantha.data.av_audio import audio_read

def read_audio_from_parquet(parquet_fp: str, row_group: int, group_index: int):
    parquet_file = read_parquet(parquet_fp)
    group = parquet_file.read_row_group(row_group).to_pandas()
    audio_bytes = group.iloc[group_index]["audio"]
    return audio_read(BytesIO(audio_bytes), format="wav")
    


In [ ]:
from IPython.display import Audio, display

audio, feature = sample_from_dfs()

with torch.no_grad():
    feature = feature.to(audio_codec.device)
    rec_audio = audio_codec.decode_z(feature[None])

display(Audio(audio[0].cpu(), rate=audio_codec.sample_rate))
display(Audio(rec_audio[0].cpu(), rate=audio_codec.sample_rate))

## Outliers

In [ ]:
from pprint import pprint
import torch
from glob import glob
from IPython.display import Audio, display

for fp in glob("*.pt"):
    print(fp)
    batch = torch.load(fp)
    
    shard_url = batch.shard_info[0].url.replace("music_Ssstk-pond5_Mnonvocal_T44k_N1608k_AudioCodec_f81b3fa_64l", "music_Ssstk-pond5_Mnonvocal_T44k_N1608k")
    row_group = batch.shard_info[0].row_group
    group_index = batch.shard_info[0].group_index
    
    audio = read_audio_from_parquet(shard_url, row_group, group_index)
    
    
    feature = batch.audio.to(audio_codec.device)
    print(feature.mean(), feature.std())
    
    with torch.no_grad():
        rec_audio = audio_codec.decode_z(feature)
    
    display(Audio(audio[0].cpu(), rate=audio_codec.sample_rate, normalize=False))
    display(Audio(rec_audio[0].cpu(), rate=audio_codec.sample_rate, normalize=True))
    # break